In [ ]:
# Install PySpark
!pip install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigData_Taxi_Fare_Prediction") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark

In [ ]:
# Download January 2023 Yellow Taxi data (~1GB)
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet

--2026-02-28 09:43:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.229, 3.170.186.41, 3.170.186.198, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.229|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47673370 (45M) [application/x-www-form-urlencoded]
Saving to: ‘yellow_tripdata_2023-01.parquet’

yellow_tripdata_202 100%[===================>]  45.46M   161MB/s    in 0.3s    

2026-02-28 09:43:49 (161 MB/s) - ‘yellow_tripdata_2023-01.parquet’ saved [47673370/47673370]



In [ ]:
df = spark.read.parquet("yellow_tripdata_2023-01.parquet")

df.printSchema()
df.count()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



3066766

In [ ]:
from pyspark.sql.functions import col, hour, dayofweek

# Remove invalid rows
df = df.filter(col("fare_amount") > 0)

# Extract time features
df = df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))
df = df.withColumn("pickup_dayofweek", dayofweek(col("tpep_pickup_datetime")))

# Select useful features
df = df.select(
    "trip_distance",
    "pickup_hour",
    "pickup_dayofweek",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "fare_amount"
)

df = df.dropna()

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "trip_distance",
        "pickup_hour",
        "pickup_dayofweek",
        "passenger_count",
        "PULocationID",
        "DOLocationID"
    ],
    outputCol="features"
)

data = assembler.transform(df).select("features", col("fare_amount").alias("label"))

In [ ]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression()

lr_model = lr.fit(train_data)

lr_predictions = lr_model.transform(test_data)

lr_predictions.select("label", "prediction").show(5)

+-----+------------------+
|label|        prediction|
+-----+------------------+
| 15.0| 17.87834084752437|
| 16.0|14.705104503664895|
| 70.0|13.016123869030013|
| 0.01|24.716396497522453|
| 15.0| 23.69277187047101|
+-----+------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor()

dt_model = dt.fit(train_data)

dt_predictions = dt_model.transform(test_data)

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(numTrees=20)

rf_model = rf.fit(train_data)

rf_predictions = rf_model.transform(test_data)

In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(maxIter=20)

gbt_model = gbt.fit(train_data)

gbt_predictions = gbt_model.transform(test_data)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(metricName="rmse")
evaluator_r2 = RegressionEvaluator(metricName="r2")

for name, predictions in [
    ("Linear Regression", lr_predictions),
    ("Decision Tree", dt_predictions),
    ("Random Forest", rf_predictions),
    ("GBT", gbt_predictions)
]:
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)
    print(f"{name} - RMSE: {rmse:.2f}, R2: {r2:.4f}")

Linear Regression - RMSE: 16.84, R2: 0.0730
Decision Tree - RMSE: 7.05, R2: 0.8375
Random Forest - RMSE: 7.83, R2: 0.7995
GBT - RMSE: 6.77, R2: 0.8505


In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

crossval = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_rmse,
    numFolds=3,
    parallelism=2
)

cv_model = crossval.fit(train_data)

In [ ]:
rf_model.write().overwrite().save("rf_taxi_model")

In [ ]:
# Small sample for sklearn baseline
sample_df = df.sample(fraction=0.05).toPandas()

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

X = sample_df.drop("fare_amount", axis=1)
y = sample_df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

sk_model = RandomForestRegressor()
sk_model.fit(X_train, y_train)

preds = sk_model.predict(X_test)

import numpy as np

mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)

print("Sklearn RMSE:", rmse)

Sklearn RMSE: 6.374187716920036


In [ ]:
# Save predictions
rf_predictions.select("label", "prediction").write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("taxi_predictions_csv")